# energy-audit · Step 3 · genre adherence (CLAP zero-shot) + quiet-clip list, all 1,800 clips

Pre-registered in `rerun/step3/GENRE_PLAN.md` (commit `7b45f37`), before any genre number existed.
Cell 1 is the scoring notebook's S1 corpus assembly, with `mrt2` added to its tag set. Cell 2 measures
per-clip CLAP genre probabilities (the scorer's checkpoint and audio path, seeded) and RMS dBFS. No new
generation; no LLM judges audio. The statistics are computed afterwards from `genre_clap_per_clip.csv`.

In [ ]:
# S1 — assemble the full corpus from 3 notebook tars + the B dataset
import subprocess, sys, os, glob, tarfile, shutil, collections, pandas as pd
subprocess.run([sys.executable,'-m','pip','install','-q','pyloudnorm'], check=True)
os.makedirs('corpus', exist_ok=True)
TAGS = {'mgs','mgm','sao','sa3s','ace','mrt2'}

for t in glob.glob('/kaggle/input/**/audio_*.tar', recursive=True):
    with tarfile.open(t) as tf:
        for m in tf.getmembers():
            b = os.path.basename(m.name)
            if b.endswith('.wav') and b.split('__')[0] in TAGS and not os.path.exists(f'corpus/{b}'):
                m.name = b; tf.extract(m, path='corpus')
for p in glob.glob('/kaggle/input/**/*.wav', recursive=True):
    b = os.path.basename(p)
    if b.split('__')[0] in TAGS and not os.path.exists(f'corpus/{b}'):
        shutil.copy(p, f'corpus/{b}')

wavs = sorted(glob.glob('corpus/*.wav'))
cnt = collections.Counter(os.path.basename(w).split('__')[0] for w in wavs)
print('corpus:', len(wavs), dict(cnt))   # want 1800: 300 per model incl. mrt2

pr = pd.read_csv(glob.glob('/kaggle/input/**/prompts.csv', recursive=True)[0])
rows = []
for w in wavs:
    mdl, pid, stail = os.path.basename(w)[:-4].split('__')
    rows.append(dict(file=w, model=mdl, prompt_id=pid, seed=int(stail[1:])))
M = pd.DataFrame(rows).merge(pr, on='prompt_id', how='left')
assert M.genre_slug.notna().all()
print(M.groupby(['model','phrasing']).size().unstack(fill_value=0))

In [ ]:
# Per-clip measurement: CLAP zero-shot genre probabilities + RMS dBFS
import numpy as np, librosa, torch, soundfile as sf, time, json
from transformers import ClapModel, ClapProcessor
clap  = ClapModel.from_pretrained('laion/clap-htsat-unfused').to('cuda').eval()
cproc = ClapProcessor.from_pretrained('laion/clap-htsat-unfused')
GENRES = {'rock': 'rock', 'edm': 'electronic dance', 'hiphop': 'hip-hop', 'folk': 'acoustic folk', 'orchestral': 'orchestral'}
SLUGS = list(GENRES); LABELS = [f'{GENRES[g]} music' for g in SLUGS]
tg = cproc(text=LABELS, return_tensors='pt', padding=True).to('cuda')
rows, t0 = [], time.time()
for k, r in enumerate(M.itertuples(), 1):
    y, sr = librosa.load(r.file, sr=None, mono=True)                 # scorer's load
    y48 = librosa.resample(y, orig_sr=sr, target_sr=48000)            # scorer's resample
    np.random.seed(0)                                                  # scorer's CLAP seed
    af = cproc(audio=y48, sampling_rate=48000, return_tensors='pt').to('cuda')
    with torch.no_grad(): lg = clap(**tg, **af).logits_per_audio[0]
    p = torch.softmax(lg.double(), dim=0).cpu().numpy()
    x, _ = sf.read(r.file, dtype='float64', always_2d=True)
    rms = float(np.sqrt(np.mean(x ** 2)))
    row = dict(file=r.file, model=r.model, prompt_id=r.prompt_id, seed=r.seed, genre_slug=r.genre_slug,
               level=r.level, phrasing=r.phrasing, top_genre=SLUGS[int(np.argmax(p))],
               rms_dbfs=20 * np.log10(rms) if rms > 0 else -np.inf)
    row.update({f'p_{g}': float(v) for g, v in zip(SLUGS, p)})
    rows.append(row)
    if k % 200 == 0: print(f'{k}/{len(M)} ({time.time()-t0:.0f}s)')
G = pd.DataFrame(rows); G.to_csv('genre_clap_per_clip.csv', index=False)
assert len(G) == 1800 and G.file.is_unique
print(G.groupby(['model', 'phrasing', 'level']).apply(lambda g: (g.top_genre == g.genre_slug).mean()).unstack().round(2))
json.dump(dict(labels=LABELS, checkpoint='laion/clap-htsat-unfused', n=len(G),
               torch=torch.__version__, gpu=torch.cuda.get_device_name(0)), open('genre_run.json', 'w'), indent=1)
import shutil; shutil.rmtree('corpus', ignore_errors=True)
